# RACE Project Runner (Colab + VSCode)

This notebook is the **single place** to run setup, preprocessing, Model A training, and quick checks.

If you are using Colab extension in VSCode, run cells top-to-bottom.

## What this notebook does
- Verifies dataset files exist in `data/raw/`
- Runs preprocessing (`src.preprocessing`)
- Trains Model A (`src.model_a_train`): classical supervised + KMeans unsupervised ensemble; reports **BLEU / ROUGE / METEOR** on generated vs reference questions
- Shows saved metrics and generated artifacts

## Expected dataset files
- `data/raw/train.csv`
- `data/raw/dev.csv` (or `data/raw/val.csv`)
- `data/raw/test.csv`

In [4]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
print('Current working directory:', ROOT)
print('Python:', sys.version)


def find_project_root(start: Path) -> Path | None:
    # 1) Walk up from current directory
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p

    # 2) Common Colab clone locations
    candidates = [
        Path('/content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project'),
        Path('/content/race_rc_project'),
    ]
    for p in candidates:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p

    # 3) Shallow search under /content for race_rc_project
    base = Path('/content')
    if base.exists():
        for p in base.rglob('race_rc_project'):
            if (p / 'requirements.txt').exists() and (p / 'src').exists():
                return p
    return None

project_root = find_project_root(ROOT)
if project_root is None:
    raise FileNotFoundError(
        'Could not find race_rc_project root. Clone the repo first, then rerun.\n'
        'Example:\n'
        '!git clone https://github.com/Hudiyahaha/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks.git\n'
        '%cd /content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project'
    )

if Path.cwd().resolve() != project_root.resolve():
    %cd {project_root}

print('Project root:', Path.cwd())

Current working directory: D:\23I-0761_23I-0765_AI\race_rc_project
Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
Project root: D:\23I-0761_23I-0765_AI\race_rc_project


In [5]:
# Install dependencies (safe to re-run)
!pip -q install -r requirements.txt

  error: subprocess-exited-with-error
  
  × Building wheel for pyarrow (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [948 lines of output]
      C:\Users\hudiyahaha\AppData\Local\Temp\pip-build-env-pq6hkgj0\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2027-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              *******************************************

## Data setup options for Colab

Use **one** of the two options below before running preprocessing:

1. Upload files directly in Colab session (temporary)
2. Copy files from Google Drive (persistent)

In [ ]:
# Option 1 (Colab): Upload CSV files from your machine into this session
# This is temporary storage; files disappear when runtime resets.

from pathlib import Path
import shutil

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
    uploaded = files.upload()  # select train.csv, dev.csv(or val.csv), test.csv
    for name in uploaded.keys():
        src = Path(name)
        dst = raw_dir / src.name
        shutil.move(str(src), str(dst))
        print('Saved:', dst)
except Exception as e:
    print('Not running in Colab or upload failed:', e)
    print('You can skip this if you will use Option 2 (Drive).')

In [ ]:
# Option 2 (Colab): Copy CSV files from Google Drive into data/raw
# Update DRIVE_DATA_DIR to your folder containing train.csv/dev.csv(or val.csv)/test.csv

from pathlib import Path
import shutil

DRIVE_DATA_DIR = '/content/drive/MyDrive/race_data'  # change this path
raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    src_dir = Path(DRIVE_DATA_DIR)
    if not src_dir.exists():
        raise FileNotFoundError(f'Drive path not found: {src_dir}')
    for fname in ['train.csv', 'dev.csv', 'val.csv', 'test.csv']:
        src = src_dir / fname
        if src.exists():
            dst = raw_dir / fname
            shutil.copy2(src, dst)
            print('Copied:', src, '->', dst)
except Exception as e:
    print('Drive copy step skipped/failed:', e)
    print('Use Option 1 or fix DRIVE_DATA_DIR and rerun this cell.')

In [ ]:
# Check dataset files
from pathlib import Path

raw_dir = Path('data/raw')
expected = ['train.csv', 'test.csv']
optional_val = ['dev.csv', 'val.csv']

for f in expected:
    print(f, '->', (raw_dir / f).exists())
print('dev/val exists ->', any((raw_dir / f).exists() for f in optional_val))

if not (raw_dir / 'train.csv').exists() or not (raw_dir / 'test.csv').exists() or not any((raw_dir / f).exists() for f in optional_val):
    raise FileNotFoundError('Missing required CSVs in data/raw. Add train/test + dev or val first.')

In [ ]:
# Run preprocessing
# Default: uses train.csv + dev/val.csv + test.csv as separate splits.
# If train and test are duplicates, use combined split (merge, dedupe by id, 80% train+val / 20% test):
# !python -m src.preprocessing --raw-dir data/raw --processed-dir data/processed --combined-split --train-fraction 0.8 --val-fraction-of-train-pool 0.1
!python -m src.preprocessing --raw-dir data/raw --processed-dir data/processed

In [ ]:
# Inspect preprocessing outputs
manifest_path = Path('data/processed/manifest.json')
if not manifest_path.exists():
    raise FileNotFoundError('manifest.json not found. Preprocessing may have failed.')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps({
    'mcq_splits': manifest.get('mcq_splits'),
    'verify_splits': manifest.get('splits'),
    'ohe_feature_dim': manifest.get('ohe_feature_dim'),
    'tfidf_feature_dim': manifest.get('tfidf_feature_dim')
}, indent=2))

In [ ]:
# Train Model A (full)
!python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional

In [ ]:
# Faster debug run (optional). Uncomment to use.
# !python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional_debug --generation-max-train-mcq 5000 --generation-max-val-mcq 2000 --generation-max-test-mcq 2000 --max-eval-mcq 500

In [ ]:
# Read Model A metrics (traditional generation: BLEU / ROUGE / METEOR — no 'results' key)
metrics_path = Path('models/model_a/traditional/metrics_summary.json')
if not metrics_path.exists():
    raise FileNotFoundError('Model A metrics file not found. Train step may have failed.')

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('Model:', metrics.get('model'))
print('Evaluation:', metrics.get('evaluation'))
print('\nConfig:')
print(json.dumps(metrics.get('config', {}), indent=2))
print('\nValidation (BLEU / ROUGE / METEOR):')
print(json.dumps(metrics.get('validation', {}), indent=2))
print('\nTest (BLEU / ROUGE / METEOR):')
print(json.dumps(metrics.get('test', {}), indent=2))
print('\nArtifacts:')
print(json.dumps(metrics.get('artifacts', {}), indent=2))

In [ ]:
# Quick artifact check
artifact_dir = Path('models/model_a/traditional')
for p in sorted(artifact_dir.glob('*')):
    print(p.name)

## Model B (traditional)

In [ ]:
# Train Model B (full)
!python -m src.model_b_train --processed-dir data/processed --output-dir models/model_b/traditional

In [ ]:
# Faster Model B debug run (optional)
# !python -m src.model_b_train --processed-dir data/processed --output-dir models/model_b/traditional_debug --max-train-mcq 5000 --max-val-mcq 2000 --max-test-mcq 2000

In [ ]:
# Read Model B metrics
metrics_b_path = Path('models/model_b/traditional/metrics_summary.json')
if not metrics_b_path.exists():
    raise FileNotFoundError('Model B metrics file not found. Train step may have failed.')

metrics_b = json.loads(metrics_b_path.read_text(encoding='utf-8'))
print('Model:', metrics_b.get('model'))
print('\nConfig:')
print(json.dumps(metrics_b.get('config', {}), indent=2))
print('\nValidation metrics:')
print(json.dumps(metrics_b.get('validation', {}), indent=2))
print('\nTest metrics:')
print(json.dumps(metrics_b.get('test', {}), indent=2))

In [ ]:
# Quick Model B artifact check
artifact_b_dir = Path('models/model_b/traditional')
for p in sorted(artifact_b_dir.glob('*')):
    print(p.name)